# 03 – Transfer Learning (ResNet-18 & MobileNetV3)
**Project:** AI Football Event Classifier (CNN) — Yannick Maas

This notebook fine-tunes two pre-trained models on the football subset:
1. **ResNet-18** — robust and accurate  
2. **MobileNetV3-Small** — lightweight and fast

Both use a multi-task head: **event** + **view** classification.

Prerequisites: `outputs/manifest.csv` (from notebook 01).

In [ ]:
import random
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
OUTPUT_DIR    = Path('outputs')
MANIFEST_PATH = OUTPUT_DIR / 'manifest.csv'
IMG_SIZE      = (224, 224)
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LR            = 3e-4
FREEZE_EPOCHS = 3   # epochs to train only the heads before unfreezing backbone

In [ ]:
df = pd.read_csv(MANIFEST_PATH)
NUM_EVENT_CLASSES = df['event_idx'].nunique()
NUM_VIEW_CLASSES  = df['view_idx'].nunique()
event_classes = sorted(df['event_label'].unique())

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}')

In [ ]:
# ImageNet-normalised transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class FootballDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(row['event_idx']), int(row['view_idx'])

train_loader = DataLoader(FootballDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(FootballDataset(val_df,   val_transform),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(FootballDataset(test_df,  val_transform),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print('DataLoaders ready')

In [ ]:
# ── Multi-task wrapper around any torchvision backbone ────────────────────────
class MultiTaskModel(nn.Module):
    def __init__(self, backbone, in_features, num_event_classes, num_view_classes):
        super().__init__()
        self.backbone   = backbone
        self.event_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_event_classes))
        self.view_head  = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_view_classes))

    def forward(self, x):
        features = self.backbone(x)
        return self.event_head(features), self.view_head(features)


def build_resnet18(num_event, num_view):
    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_features = backbone.fc.in_features
    backbone.fc = nn.Identity()  # remove original head
    return MultiTaskModel(backbone, in_features, num_event, num_view)


def build_mobilenet_v3_small(num_event, num_view):
    backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
    in_features = backbone.classifier[3].in_features
    backbone.classifier[3] = nn.Identity()  # remove original head
    return MultiTaskModel(backbone, in_features, num_event, num_view)


print('Model builders defined')

In [ ]:
def train_model(model, model_name, freeze_backbone_epochs=3):
    """Full training loop with optional backbone freezing phase."""
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'val_event_acc': [], 'val_view_acc': []}
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, NUM_EPOCHS + 1):
        # Freeze/unfreeze backbone
        frozen = (epoch <= freeze_backbone_epochs)
        for param in model.backbone.parameters():
            param.requires_grad = not frozen
        head_params = list(model.event_head.parameters()) + list(model.view_head.parameters())
        back_params = [p for p in model.backbone.parameters() if p.requires_grad]
        optimizer = optim.Adam(
            [{'params': head_params, 'lr': LR},
             {'params': back_params, 'lr': LR / 10}],
        ) if back_params else optim.Adam(head_params, lr=LR)

        # Train
        model.train()
        tl = ev_c = vw_c = n = 0
        for imgs, ev, vw in train_loader:
            imgs, ev, vw = imgs.to(DEVICE), ev.to(DEVICE), vw.to(DEVICE)
            e_out, v_out = model(imgs)
            loss = criterion(e_out, ev) + criterion(v_out, vw)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tl += loss.item() * len(imgs); n += len(imgs)
        train_loss = tl / n

        # Validate
        model.eval()
        vl = ev_c = vw_c = n = 0
        with torch.no_grad():
            for imgs, ev, vw in val_loader:
                imgs, ev, vw = imgs.to(DEVICE), ev.to(DEVICE), vw.to(DEVICE)
                e_out, v_out = model(imgs)
                loss = criterion(e_out, ev) + criterion(v_out, vw)
                vl   += loss.item() * len(imgs)
                ev_c += (e_out.argmax(1) == ev).sum().item()
                vw_c += (v_out.argmax(1) == vw).sum().item()
                n    += len(imgs)
        val_loss = vl / n
        ev_acc = ev_c / n; vw_acc = vw_c / n

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_event_acc'].append(ev_acc)
        history['val_view_acc'].append(vw_acc)

        phase = '(frozen)' if frozen else '(fine-tune)'
        print(f'[{model_name}] Epoch {epoch:2d}/{NUM_EPOCHS} {phase}  '
              f'train={train_loss:.4f}  val={val_loss:.4f}  '
              f'event_acc={ev_acc:.3f}  view_acc={vw_acc:.3f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    torch.save(best_state, OUTPUT_DIR / f'{model_name}.pth')
    print(f'Best model saved → outputs/{model_name}.pth  (val_loss={best_val_loss:.4f})')
    return model, history

In [ ]:
# ── Train ResNet-18 ───────────────────────────────────────────────────────────
resnet_model = build_resnet18(NUM_EVENT_CLASSES, NUM_VIEW_CLASSES)
resnet_model, resnet_history = train_model(resnet_model, 'resnet18', FREEZE_EPOCHS)

In [ ]:
# ── Train MobileNetV3-Small ───────────────────────────────────────────────────
mobilenet_model = build_mobilenet_v3_small(NUM_EVENT_CLASSES, NUM_VIEW_CLASSES)
mobilenet_model, mobilenet_history = train_model(mobilenet_model, 'mobilenet_v3_small', FREEZE_EPOCHS)

In [ ]:
# ── Compare training curves ───────────────────────────────────────────────────
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, resnet_history['val_loss'],   label='ResNet-18')
axes[0].plot(epochs, mobilenet_history['val_loss'],label='MobileNetV3-Small')
axes[0].set_title('Validation Loss'); axes[0].legend()

axes[1].plot(epochs, resnet_history['val_event_acc'],   label='ResNet-18')
axes[1].plot(epochs, mobilenet_history['val_event_acc'],label='MobileNetV3-Small')
axes[1].set_title('Val Event Accuracy'); axes[1].legend()

axes[2].plot(epochs, resnet_history['val_view_acc'],    label='ResNet-18')
axes[2].plot(epochs, mobilenet_history['val_view_acc'], label='MobileNetV3-Small')
axes[2].set_title('Val View Accuracy'); axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'transfer_learning_curves.png', dpi=120)
plt.show()

In [ ]:
def evaluate_on_test(model, model_name):
    model.eval()
    all_et, all_ep = [], []
    with torch.no_grad():
        for imgs, ev, _ in test_loader:
            e_out, _ = model(imgs.to(DEVICE))
            all_et.extend(ev.numpy())
            all_ep.extend(e_out.argmax(1).cpu().numpy())

    print(f'\n=== {model_name} — Test Set ===')
    print(classification_report(all_et, all_ep, target_names=event_classes))

    cm = confusion_matrix(all_et, all_ep)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=event_classes, yticklabels=event_classes)
    plt.title(f'Confusion Matrix – {model_name}')
    plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{model_name}_confusion_matrix.png', dpi=120)
    plt.show()

evaluate_on_test(resnet_model,    'resnet18')
evaluate_on_test(mobilenet_model, 'mobilenet_v3_small')

## Summary & Next Steps

| Model | Val Event Acc | Val View Acc |
|-------|--------------|-------------|
| Baseline CNN | see nb 02 | see nb 02 |
| ResNet-18 | — | — |
| MobileNetV3-Small | — | — |

Fill in the table after running the notebooks.

**Next steps toward the full vision:**
- Train on the full dataset once performance is validated
- Extend to video by running the classifier on sampled frames
- Use event predictions to auto-cut highlight clips

➡️ Continue with **`04_model_comparison.ipynb`** for side-by-side evaluation.